# 06 — Régimes de marché & Backtest agriculteur

**Questions centrales :**
1. Le marché du maïs a-t-il des régimes détectables ?
2. Notre système de conseil améliore-t-il la rentabilité agriculteur ?

## Pourquoi les régimes ?

Un seul modèle entraîné sur 25 ans suppose implicitement que le marché se comporte toujours pareil.
Or :
- **2008** : bulle commodités (bull)
- **2012** : sécheresse US (bull) 
- **2014-2019** : oversupply, prix bas (range-bear)
- **2020-2022** : COVID + Ukraine (bull)
- **2023-2025** : normalisation (range)

Si on peut **détecter le régime en temps réel**, on peut adapter les seuils de décision.

In [ ]:
import sys
sys.path.insert(0, '../../src')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
from pathlib import Path

ROOT = Path('../../')
plt.style.use('seaborn-v0_8-whitegrid')

reg = pd.read_parquet(ROOT / 'artefacts/professional_study/regime_timeseries.parquet')
reg['Date'] = pd.to_datetime(reg['Date'])

bk_yr  = pd.read_parquet(ROOT / 'artefacts/farmer_backtest/yearly_results.parquet')
bk_day = pd.read_parquet(ROOT / 'artefacts/farmer_backtest/daily_decisions.parquet')
bk_sm  = json.loads((ROOT / 'artefacts/farmer_backtest/summary.json').read_text())

ss = json.loads((ROOT / 'artefacts/professional_study/study_summary.json').read_text())

print('Régimes columns :', list(reg.columns))
print('Régimes obs :', len(reg))
if 'regime' in reg.columns:
    print('Distribution :', reg['regime'].value_counts().to_dict())

## 1. Régimes dans le temps

In [ ]:
regime_colors = {'bull': '#4CAF50', 'range': '#2196F3', 'bear': '#F44336'}

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

# Graphe 1 : prix du maïs coloré par régime
if 'corn_close' in reg.columns and 'regime' in reg.columns:
    for regime, color in regime_colors.items():
        mask = reg['regime'] == regime
        axes[0].scatter(reg.loc[mask, 'Date'], reg.loc[mask, 'corn_close'],
                        c=color, s=2, alpha=0.6, label=regime)
    axes[0].set_title('Prix maïs CBOT coloré par régime détecté (Markov-switching)', fontsize=11)
    axes[0].set_ylabel('Prix (cts/bu)')
    axes[0].legend(markerscale=5)

# Graphe 2 : score de régime
if 'regime_score' in reg.columns:
    axes[1].plot(reg['Date'], reg['regime_score'], lw=0.8, color='purple')
    axes[1].axhline(0, color='gray', lw=0.5)
    axes[1].fill_between(reg['Date'], reg['regime_score'], 0,
                          where=reg['regime_score']>0, alpha=0.3, color='green', label='Bull')
    axes[1].fill_between(reg['Date'], reg['regime_score'], 0,
                          where=reg['regime_score']<0, alpha=0.3, color='red', label='Bear')
    axes[1].set_title('Score de régime (positif = bull, négatif = bear)', fontsize=11)
    axes[1].set_ylabel('Score')
    axes[1].legend()

# Graphe 3 : volatilité réalisée
if 'realized_vol_60d' in reg.columns:
    axes[2].plot(reg['Date'], reg['realized_vol_60d'] * 100, lw=0.8, color='orange')
    axes[2].set_title('Volatilité réalisée 60j annualisée (%)', fontsize=11)
    axes[2].set_ylabel('Vol (%)')
elif 'return_60d' in reg.columns:
    axes[2].plot(reg['Date'], reg['return_60d'] * 100, lw=0.8, color='orange')
    axes[2].set_title('Rendement 60j (%)', fontsize=11)

plt.tight_layout()
plt.show()

## 2. Problème actuel : régime bear quasi-inexistant

Le résultat Markov-switching donne :
- **range : 90%** des observations
- **bull : 9.9%**
- **bear : 0.05%** (3 jours seulement !)

Avec seulement 3 jours bear sur 25 ans, le régime bear est **inutilisable**.

**Pourquoi ce résultat ?**

Sur 2000-2025, le maïs n'a pas vraiment connu de bear market prolongé (pas comme les actions). Les baisses ont été brutales et courtes, pas prolongées. Le Markov 3 états ne capture pas ça bien.

**Ce qu'on va tester à la place :**
1. Markov 2 états (bull/range-bear)
2. Régimes rule-based : bull si return_90d > 20%, bear si return_90d < -20%, range sinon
3. Régimes par volatilité : high vol / low vol

In [ ]:
# Alternative : régime rule-based sur rendement 90j
if 'return_60d' in reg.columns:
    ret_col = 'return_60d'
elif 'corn_close' in reg.columns:
    reg['return_60d'] = reg['corn_close'].pct_change(60)
    ret_col = 'return_60d'
else:
    ret_col = None

if ret_col:
    # Rule-based 3 états
    conditions = [
        reg[ret_col] > 0.20,
        reg[ret_col] < -0.20,
    ]
    reg['regime_rule'] = 'range'
    reg.loc[conditions[0], 'regime_rule'] = 'bull'
    reg.loc[conditions[1], 'regime_rule'] = 'bear'
    
    print('Distribution régimes rule-based (rendement 60j) :')
    vc = reg['regime_rule'].value_counts()
    for r, cnt in vc.items():
        print(f'  {r:10s}: {cnt:5d} obs  ({cnt/len(reg):.1%})')
    
    print()
    print('Distribution régimes Markov-switching :')
    if 'regime' in reg.columns:
        vc2 = reg['regime'].value_counts()
        for r, cnt in vc2.items():
            print(f'  {r:10s}: {cnt:5d} obs  ({cnt/len(reg):.1%})')

## 3. Performance des modèles par régime

Un modèle peut être très bon en régime range et mauvais en régime bull. C'est essentiel à mesurer.

In [ ]:
# On fusionne prédictions + régimes
pred_path = ROOT / 'artefacts/professional_study/model_predictions.parquet'
if pred_path.exists():
    preds = pd.read_parquet(pred_path)
    preds['Date'] = pd.to_datetime(preds['Date']) if 'Date' in preds.columns else None
    
    print('Colonnes prédictions :', list(preds.columns)[:20])
    
    if 'Date' in preds.columns and 'regime' in reg.columns:
        merged = preds.merge(reg[['Date','regime']], on='Date', how='left')
        
        print(f'\nMerge : {len(merged):,} lignes, régimes présents : {merged["regime"].notna().sum():,}')
else:
    print('Fichier model_predictions.parquet non disponible.')

## 4. Backtest agriculteur

**Scénario :** agriculteur en Iowa, 50 000 bushels à vendre, stockage possible à 0.04 USD/bu/mois.

**Stratégies comparées :**
- `sell_at_harvest_100` — tout vendre à la récolte (octobre)
- `sell_dca_monthly` — vendre 1/12 chaque mois (DCA)
- `model_adviser` — suivre les signaux du modèle

In [ ]:
strategies = bk_sm.get('strategies', [])
if strategies:
    strat_df = pd.DataFrame(strategies)
    print('Résultats backtest agriculteur (2015-2025, Iowa) :')
    print(strat_df[['strategy','avg_revenue_per_bu','pct_years_beating_harvest_only',
                     'sharpe_per_year']].to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    colors = ['#d9534f' if 'harvest' in s else ('#5bc0de' if 'dca' in s else '#5cb85c')
              for s in strat_df['strategy']]
    
    # Revenu moyen
    axes[0].bar(strat_df['strategy'], strat_df['avg_revenue_per_bu'], color=colors, alpha=0.85)
    axes[0].set_title('Revenu moyen par bushel (USD/bu)\n2015-2025, Iowa')
    axes[0].set_ylabel('USD/bu')
    axes[0].set_ylim(3.8, max(strat_df['avg_revenue_per_bu']) * 1.05)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20, ha='right')
    for i, (_, row) in enumerate(strat_df.iterrows()):
        axes[0].annotate(f"{row['avg_revenue_per_bu']:.3f}",
                        xy=(i, row['avg_revenue_per_bu'] + 0.01), ha='center', fontsize=9)
    
    # % années gagnantes vs. récolte
    axes[1].bar(strat_df['strategy'], strat_df['pct_years_beating_harvest_only'] * 100,
                color=colors, alpha=0.85)
    axes[1].axhline(50, color='gray', lw=1.5, ls='--')
    axes[1].set_title('% années battant la vente à la récolte')
    axes[1].set_ylabel('%')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha='right')
    
    plt.tight_layout()
    plt.show()

## 5. Résultats par année

In [ ]:
print('Colonnes yearly :', list(bk_yr.columns))
bk_yr.head()

In [ ]:
if not bk_yr.empty:
    # Chercher colonnes revenue par stratégie
    rev_cols = [c for c in bk_yr.columns if 'revenue' in c.lower() or 'price' in c.lower() or 'usd' in c.lower()]
    yr_col = next((c for c in bk_yr.columns if 'year' in c.lower()), None)
    
    if yr_col and rev_cols:
        fig, ax = plt.subplots(figsize=(16, 6))
        for col in rev_cols[:4]:
            ax.plot(bk_yr[yr_col], bk_yr[col], marker='o', lw=1.5, label=col)
        ax.set_title('Revenu par bushel par année et par stratégie')
        ax.set_xlabel('Année')
        ax.set_ylabel('USD/bu')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('Colonnes disponibles :', list(bk_yr.columns))
        print(bk_yr.head(10).to_string())

## 6. Lecture honnête du backtest

### Ce que les résultats disent

```
DCA mensuel      : 4.30 USD/bu  (bat la récolte 7 ans sur 10)
Conseil modèle   : 4.16 USD/bu  (bat la récolte 5 ans sur 10)
Vente à récolte  : 4.16 USD/bu  (référence)
```

### Interprétation

**Le DCA mensuel est la vraie gagnante.** Cela confirme un principe financier classique :
en marché incertain, étaler les ventes réduit le risque sans intelligence particulière.

**Le conseil modèle ne bat pas clairement la récolte.** Ce n'est pas une défaite — c'est une réalité honnête.
Le modèle ne perd pas non plus. Il est à égalité, mais sans la simplicité du DCA.

**Pourquoi le modèle ne bat pas mieux ?**
1. Il déclenche souvent SELL_THIRDS (incertitude) → comportement DCA-like sans optimisation
2. La DA à h=20j est 55-61% → pas assez fort pour parier
3. Les coûts de stockage (0.04$/bu/mois) pénalisent le stockage spéculatif

### Ce qu'il faudrait pour un vrai gain

1. **DA > 63-65%** à h=20j → possible avec Crop Progress + EIA éthanol + Drought Monitor
2. **Décision par prix cible** (pas seulement direction) → si Q90 > prix actuel × 1.10, stocker
3. **Calibration par régime** → en bull, attendre plus longtemps

**→ Prochain carnet :** Conclusions et feuille de route.